In [ ]:
"""
PASSO 2 — Cálculo dos 44 indicadores técnicos + Label.

Para cada {Code}_New.csv (OHLCV ajustado), calcula 44 indicadores técnicos
e o Label binário (1 se retorno do dia seguinte >= 0, senão 0).
Salva como {Code}_TechIndi.csv com colunas: Date, 44 features, Label.


Uso:
"""

from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm

# ===================== CONFIGURAÇÃO =====================
BASE_DIR = Path(r"C:\Users\Rodrigo\Desktop\Replicando para B3_2\B3ICS")
SEC_NAMES = BASE_DIR / ".NewB3_pruned.csv"
K_LABEL = 1  # k=1: label baseado no retorno do dia seguinte
# ========================================================


# ============================================================
#                    FUNÇÕES AUXILIARES
# ============================================================

def sma(series, n):
    return series.rolling(window=n, min_periods=n).mean()

def ema(series, n):
    return series.ewm(span=n, adjust=False).mean()

def true_range(high, low, close):
    """True Range para cálculo do ATR."""
    prev_close = close.shift(1)
    tr1 = high - low
    tr2 = (high - prev_close).abs()
    tr3 = (low - prev_close).abs()
    return pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)


# ============================================================
#              44 INDICADORES TÉCNICOS
# ============================================================

def calc_atr(high, low, close, n=14):
    """1. ATR - Average True Range (SMA do True Range)."""
    tr = true_range(high, low, close)
    return sma(tr, n).rename("ATR")

def calc_adx(high, low, close, n=14):
    """2. ADX - Average Directional Index."""
    prev_high = high.shift(1)
    prev_low = low.shift(1)
    plus_dm = np.where((high - prev_high) > (prev_low - low),
                       np.maximum(high - prev_high, 0), 0)
    minus_dm = np.where((prev_low - low) > (high - prev_high),
                        np.maximum(prev_low - low, 0), 0)
    tr = true_range(high, low, close)
    atr_n = sma(tr, n)
    plus_di = 100 * sma(pd.Series(plus_dm, index=high.index), n) / (atr_n + 1e-10)
    minus_di = 100 * sma(pd.Series(minus_dm, index=high.index), n) / (atr_n + 1e-10)
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di + 1e-10)
    return sma(dx, n).rename("ADX")

def calc_obv(close, volume):
    """3. OBV - On Balance Volume."""
    direction = np.sign(close.diff()).fillna(0)
    return (direction * volume).cumsum().rename("OBV")

def calc_wr(high, low, close, n=14):
    """4. WR - Williams %R."""
    hh = high.rolling(n).max()
    ll = low.rolling(n).min()
    wr = (hh - close) / (hh - ll + 1e-10) * (-100)
    return wr.rename("WR")

def calc_rsi(close, n=14):
    """5. RSI - Relative Strength Index."""
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1/n, min_periods=n, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/n, min_periods=n, adjust=False).mean()
    rs = avg_gain / (avg_loss + 1e-10)
    return (100 - 100 / (1 + rs)).rename("RSI")

def calc_cmf(high, low, close, volume, n=20):
    """6. CMF - Chaikin Money Flow."""
    mfm = ((close - low) - (high - close)) / (high - low + 1e-10)
    mfv = mfm * volume
    return (mfv.rolling(n).sum() / (volume.rolling(n).sum() + 1e-10)).rename("CMF")

def calc_bbands(high, low, close, n=20, ndev=2):
    """7-8. Bollinger Bands: BandPer (%b) e BandWid."""
    # TTR::BBands usa HLC, aqui usamos typical price
    tp = (high + low + close) / 3
    ma = sma(tp, n)
    sd = tp.rolling(n).std()
    bb_up = ma + ndev * sd
    bb_dn = ma - ndev * sd
    bandper = ((close - bb_dn) / (bb_up - bb_dn + 1e-10)).rename("BandPer")
    bandwid = ((bb_up - bb_dn) / (ma + 1e-10)).rename("BandWid")
    return bandper, bandwid

def calc_chaikin_osc(high, low, close, volume, ema_short=3, ema_long=10):
    """9. CO - Chaikin Oscillator (EMA3 - EMA10 da A/D Line)."""
    H, L, C, V = high.values, low.values, close.values, volume.values
    N = len(H)
    ad = np.zeros(N)
    eps = 1e-10
    ad[0] = ((C[0] - L[0]) - (H[0] - C[0])) * V[0] / (H[0] - L[0] + eps)
    for i in range(1, N):
        clv = ((C[i] - L[i]) - (H[i] - C[i])) / (H[i] - L[i] + eps)
        ad[i] = ad[i - 1] + clv * V[i]
    ad_s = pd.Series(ad, index=close.index)
    return (ema(ad_s, ema_short) - ema(ad_s, ema_long)).rename("CO")

def calc_disparity(close, n=20):
    """10. DIS - Disparity."""
    return ((close / (sma(close, n) + 1e-10)) * 100).rename("DIS")

def calc_eom(high, low, volume):
    """11. EOM - Ease of Movement."""
    H, L, V = high.values, low.values, volume.values
    N = len(H)
    eom = np.zeros(N)
    eom[0] = (H[0] + L[0]) / 2.0
    for i in range(1, N):
        eom[i] = ((H[i] + L[i]) / 2 - (H[i-1] + L[i-1]) / 2) * (H[i] - L[i]) / (V[i] + 1e-10)
    return pd.Series(eom, index=high.index).rename("EOM")

def calc_force_index(close, volume):
    """12. FI - Force Index (SMA2)."""
    C, V = close.values, volume.values
    N = len(C)
    fi = np.zeros(N)
    fi[0] = C[0]
    for i in range(1, N):
        fi[i] = (C[i] - C[i-1]) * V[i]
    return sma(pd.Series(fi, index=close.index), 2).rename("FI")

def calc_mao(close):
    """13. MAO - MA Oscillator (SMA12 - SMA26)."""
    return (sma(close, 12) - sma(close, 26)).rename("MAO")

def calc_mfi(high, low, close, volume, n=14):
    """14. MFI - Money Flow Index."""
    tp = (high + low + close) / 3
    mf = tp * volume
    delta = tp.diff()
    pos_mf = pd.Series(np.where(delta > 0, mf, 0), index=close.index)
    neg_mf = pd.Series(np.where(delta <= 0, mf, 0), index=close.index)
    pos_sum = pos_mf.rolling(n).sum()
    neg_sum = neg_mf.rolling(n).sum()
    mfi = 100 - 100 / (1 + pos_sum / (neg_sum + 1e-10))
    return mfi.rename("MFI")

def calc_mass_index(high, low, n=9, sum_n=9):
    """15. MI - Mass Index."""
    r = high - low
    ema1 = ema(r, n)
    ema2 = ema1 ** 2
    x = ema1 / (ema2 + 1e-10)
    return x.rolling(sum_n).sum().rename("MI")

def calc_momentum(close, n=10):
    """16. MOM - Momentum."""
    C = close.values
    N = len(C)
    mom = np.full(N, np.nan)
    for i in range(n - 1, N):
        mom[i] = (C[i] / C[i - (n - 1)]) * 100
    return pd.Series(mom, index=close.index).rename("MOM")

def calc_nco(close, n=12):
    """17. NCO - Net Change Oscillator (momentum de n períodos)."""
    return close.diff(n).rename("NCO")

def calc_po(close):
    """18. PO - Price Oscillator."""
    sma5 = sma(close, 5)
    sma10 = sma(close, 10)
    return ((sma5 - sma10) / (sma5 + 1e-10)).rename("PO")

def calc_psy(close, n=10):
    """19. PSY - Psychology Line."""
    C = close.values
    N = len(C)
    psy = np.full(N, np.nan)
    for i in range(n, N):
        window = C[i - n : i + 1]
        diffs = np.diff(window)
        psy[i] = np.sum(diffs >= 0) / n
    return pd.Series(psy, index=close.index).rename("PSY")

def calc_rmi(close, length=14):
    """20. RMI - Relative Momentum Index."""
    C = close.values
    N = len(C)
    mo = np.diff(C, prepend=np.nan)  # momentum(C, 1)
    rmi = np.full(N, np.nan)
    eps = 0.01
    for i in range(length - 1, N):
        window = mo[i - (length - 1) : i + 1]
        window = window[~np.isnan(window)]
        pos = np.sum(window[window >= 0])
        neg = np.sum(np.abs(window[window < 0]))
        rmi[i] = pos / (pos + neg + eps)
    return pd.Series(rmi, index=close.index).rename("RMI")

def calc_roc(close, n=12):
    """21. ROC - Rate of Change."""
    return close.pct_change(n).rename("ROC")

def calc_sroc(close):
    """22. SROC - Smoothed Rate of Change."""
    return ((ema(close, 20) / (ema(close, 10) + 1e-10)) * 100).rename("SROC")

def calc_sonar(close):
    """23-24. SONAR e SONSIG."""
    ema25 = ema(close, 25)
    sonar = ema25.diff(25).rename("SONAR")
    sonsig = ema(sonar, 9).rename("SONSIG")
    return sonar, sonsig

def calc_trix(close, n=12):
    """25. TRIX."""
    e1 = ema(close, n)
    e2 = ema(e1, n)
    e3 = ema(e2, n)
    return e3.pct_change().rename("TRIX")

def calc_vma(volume, n=20):
    """26. VMA - Volume Moving Average."""
    return sma(volume, n).rename("VMA")

def calc_vos(volume):
    """27. VOS - Volume Oscillator."""
    vm = ema(volume, 12)
    vn = ema(volume, 26)
    return (((vm - vn) / (vn + 1e-10)) * 100).rename("VOS")

def calc_vroc(volume, n=14):
    """28. VROC - Volume Rate of Change."""
    return volume.pct_change(n).rename("VROC")

def calc_return_sigma(close, n=14):
    """29-30. Return (média móvel do retorno) e Sigma (desvio-padrão móvel)."""
    ret = close.pct_change()
    return_ma = ret.rolling(n).mean().rename("Return")
    sigma = ret.rolling(n).std().rename("Sigma")
    return return_ma, sigma

def calc_cci(high, low, close, n=14):
    """31. CCI - Commodity Channel Index."""
    tp = (high + low + close) / 3
    ma = sma(tp, n)
    md = tp.rolling(n).apply(lambda x: np.mean(np.abs(x - x.mean())), raw=True)
    return ((tp - ma) / (0.015 * md + 1e-10)).rename("CCI")

def calc_kdj(high, low, close):
    """32-35. RSV, Kvalue, Dvalue, Jvalue (Stochastic KDJ)."""
    n = 14
    hh = high.rolling(n).max()
    ll = low.rolling(n).min()
    rsv = ((close - ll) / (hh - ll + 1e-10)).rename("RSV")
    # K = EMA(RSV, ratio=1/3) ~ equivale a ewm(alpha=1/3)
    kvalue = rsv.ewm(alpha=1/3, adjust=False).mean().rename("Kvalue")
    dvalue = kvalue.ewm(alpha=1/3, adjust=False).mean().rename("Dvalue")
    jvalue = (3 * kvalue - 2 * dvalue).rename("Jvalue")
    return rsv, kvalue, dvalue, jvalue

def calc_macd(close, nfast=12, nslow=26, nsig=9):
    """36. MACD."""
    macd_line = ema(close, nfast) - ema(close, nslow)
    return macd_line.rename("MACD")

def calc_chaikin_ad(high, low, close, volume):
    """37. AD - Chaikin Accumulation/Distribution Line."""
    H, L, C, V = high.values, low.values, close.values, volume.values
    N = len(H)
    ad = np.zeros(N)
    eps = 1e-10
    ad[0] = ((C[0] - L[0]) - (H[0] - C[0])) * V[0] / (H[0] - L[0] + eps)
    for i in range(1, N):
        clv = ((C[i] - L[i]) - (H[i] - C[i])) / (H[i] - L[i] + eps)
        ad[i] = ad[i - 1] + clv * V[i]
    return pd.Series(ad, index=close.index).rename("AD")

def calc_chaikin_vol(high, low, n=10):
    """38. VOLA - Chaikin Volatility."""
    hl = high - low
    ema_hl = ema(hl, n)
    return ema_hl.pct_change(n).rename("VOLA")

def calc_nbias(close, n=6):
    """39. NBIAS."""
    sma_n = sma(close, n)
    return (100 * (close - sma_n) / (sma_n + 1e-10)).rename("NBIAS")

def calc_ret(close):
    """40. Ret - Retorno diário (ROC de 1 período)."""
    return close.pct_change().rename("Ret")

def calc_moving_averages(close):
    """41-44. SMA_5, SMA_10, EMA_5, EMA_10."""
    return (
        sma(close, 5).rename("SMA_5"),
        sma(close, 10).rename("SMA_10"),
        ema(close, 5).rename("EMA_5"),
        ema(close, 10).rename("EMA_10"),
    )


# ============================================================
#                    LABEL
# ============================================================

def calc_label(close, k=1):
    """
    Label binário: equivalente à label_func do R.
    Para cada dia t, calcula a média dos log-retornos dos próximos k dias.
    Se média >= 0: Label = 1, senão Label = 0.
    """
    log_ret = np.log(close / close.shift(1))
    labels = pd.Series(np.nan, index=close.index)
    vals = log_ret.values
    N = len(vals)
    for i in range(0, N - k):
        mean_next_k = np.nanmean(vals[i + 1 : i + 1 + k])
        labels.iloc[i] = 1 if mean_next_k >= 0 else 0
    return labels.rename("Label")


# ============================================================
#              FUNÇÃO PRINCIPAL: 44 INDICADORES + LABEL
# ============================================================

def compute_all_indicators(filepath: Path) -> pd.DataFrame:
    """
    Lê um arquivo {Code}_New.csv e retorna DataFrame com
    Date + 44 indicadores + Label.
    """
    df = pd.read_csv(filepath)
    df.columns = [c.strip() for c in df.columns]

    # Identificar colunas
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).sort_values(date_col).reset_index(drop=True)

    # Mapear OHLCV (robusto a variações de nome)
    col_map = {}
    for c in df.columns:
        cl = c.lower()
        if "open" in cl:   col_map["Open"] = c
        elif "high" in cl: col_map["High"] = c
        elif "low" in cl:  col_map["Low"] = c
        elif "close" in cl and "adj" not in cl: col_map["Close"] = c
        elif "vol" in cl:  col_map["Volume"] = c

    for needed in ["Open", "High", "Low", "Close", "Volume"]:
        if needed not in col_map:
            raise ValueError(f"Coluna {needed} não encontrada em {filepath.name}")

    O = df[col_map["Open"]].astype(float)
    H = df[col_map["High"]].astype(float)
    L = df[col_map["Low"]].astype(float)
    C = df[col_map["Close"]].astype(float)
    V = df[col_map["Volume"]].astype(float)

    # Filtrar linhas com Low=0 (equivalente ao R: Newstock[Newstock$Low>0,])
    valid = L > 0
    O, H, L, C, V = O[valid], H[valid], L[valid], C[valid], V[valid]
    dates = df.loc[valid, date_col]

    # Recalcular índices
    O = O.reset_index(drop=True)
    H = H.reset_index(drop=True)
    L = L.reset_index(drop=True)
    C = C.reset_index(drop=True)
    V = V.reset_index(drop=True)
    dates = dates.reset_index(drop=True)

    # --- Calcular os 44 indicadores ---
    bandper, bandwid = calc_bbands(H, L, C)
    sonar, sonsig = calc_sonar(C)
    return_ma, sigma = calc_return_sigma(C)
    rsv, kval, dval, jval = calc_kdj(H, L, C)
    sma5, sma10, ema5, ema10 = calc_moving_averages(C)

    indicators = pd.concat([
        calc_atr(H, L, C, 14),       # 1
        calc_adx(H, L, C, 14),       # 2
        calc_obv(C, V),              # 3
        calc_wr(H, L, C, 14),        # 4
        calc_rsi(C, 14),             # 5
        calc_cmf(H, L, C, V, 20),    # 6
        bandper,                      # 7
        bandwid,                      # 8
        calc_chaikin_osc(H, L, C, V), # 9
        calc_disparity(C, 20),        # 10
        calc_eom(H, L, V),           # 11
        calc_force_index(C, V),       # 12
        calc_mao(C),                 # 13
        calc_mfi(H, L, C, V, 14),    # 14
        calc_mass_index(H, L),        # 15
        calc_momentum(C, 10),         # 16
        calc_nco(C, 12),             # 17
        calc_po(C),                  # 18
        calc_psy(C, 10),             # 19
        calc_rmi(C, 14),             # 20  (era comentado no R)
        calc_roc(C, 12),             # 21
        calc_sroc(C),                # 22
        sonar,                        # 23
        sonsig,                       # 24
        calc_trix(C, 12),            # 25
        calc_vma(V, 20),             # 26
        calc_vos(V),                 # 27
        calc_vroc(V, 14),            # 28  (era comentado no R)
        return_ma,                    # 29
        sigma,                        # 30
        calc_cci(H, L, C, 14),       # 31
        rsv,                          # 32
        kval,                         # 33
        dval,                         # 34
        jval,                         # 35
        calc_macd(C),                # 36
        calc_chaikin_ad(H, L, C, V), # 37
        calc_chaikin_vol(H, L, 10),  # 38
        calc_nbias(C, 6),            # 39
        calc_ret(C),                 # 40
        sma5,                         # 41
        sma10,                        # 42
        ema5,                         # 43
        ema10,                        # 44
    ], axis=1)

    # Label
    label = calc_label(C, K_LABEL)

    # Montar DataFrame final
    out = pd.DataFrame(index=dates.index)
    out["Date"] = dates.values
    for col in indicators.columns:
        out[col] = indicators[col].values
    out["Label"] = label.values

    # Remover linhas com NaN (equivalente a na.omit no R)
    out = out.dropna().reset_index(drop=True)

    return out


# ============================================================
#                    EXECUÇÃO
# ============================================================

def main():
    codes_df = pd.read_csv(SEC_NAMES, dtype=str, encoding="utf-8-sig")
    codes_df.columns = [c.strip() for c in codes_df.columns]
    codes = codes_df["Code"].str.strip().str.upper().tolist()

    print(f"Total de tickers: {len(codes)}")
    print(f"Indicadores: 44 + Label\n")

    report = []
    for code in tqdm(codes, desc="Calculando indicadores"):
        infile = BASE_DIR / f"{code}_New.csv"
        outfile = BASE_DIR / f"{code}_TechIndi.csv"

        if not infile.exists():
            report.append({"Code": code, "status": "no_New_file", "rows": 0})
            continue

        if outfile.exists():
            report.append({"Code": code, "status": "skipped", "rows": 0})
            continue

        try:
            result = compute_all_indicators(infile)

            if len(result) < 100:
                report.append({"Code": code, "status": f"too_few_rows ({len(result)})", "rows": len(result)})
                continue

            # Verificar que temos 44 features + Date + Label = 46 colunas
            assert result.shape[1] == 46, f"Esperado 46 colunas, obteve {result.shape[1]}"

            result.to_csv(outfile, index=False, encoding="utf-8-sig")
            report.append({"Code": code, "status": "ok", "rows": len(result)})

        except Exception as e:
            report.append({"Code": code, "status": f"error: {e}", "rows": 0})

    # Relatório
    report_df = pd.DataFrame(report)
    report_df.to_csv(BASE_DIR / "techindi_report.csv", index=False, encoding="utf-8-sig")

    n_ok = (report_df["status"] == "ok").sum()
    n_skip = (report_df["status"] == "skipped").sum()
    n_fail = len(report_df) - n_ok - n_skip
    print(f"\n{'='*50}")
    print(f"Concluído: {n_ok} gerados, {n_skip} já existiam, {n_fail} falharam.")
    print(f"Colunas por arquivo: Date + 44 indicadores + Label = 46")
    print(f"Relatório: techindi_report.csv")


if __name__ == "__main__":
    main()

Total de tickers: 246
Indicadores: 44 + Label



Calculando indicadores: 100%|██████████| 246/246 [00:00<00:00, 1063.45it/s]


Concluído: 0 gerados, 246 já existiam, 0 falharam.
Colunas por arquivo: Date + 44 indicadores + Label = 46
Relatório: techindi_report.csv
